In [ ]:
%matplotlib widget

In [ ]:
import hyperspy.api as hs
import pyxem as pxm
import matplotlib.pyplot as plt
import numpy as np
from scipy.interpolate import RegularGridInterpolator
from scipy.optimize import root

# Load data

In [ ]:
s = hs.load("data/2.zspy", lazy=True)
s_bs = s.get_direct_beam_position(method="center_of_mass")
s_bs.compute()
s_bs_lp = s_bs.get_linear_plane()
s_bs_corr = s_bs-s_bs_lp

In [ ]:
fig, ax = plt.subplots()

In [ ]:
s_bs_corr_crop = s_bs_corr.inav[14.:18., 20.:24.]
s_bs_corr_crop = (
    s_bs_corr_crop.isig[0].data
    + 1j * s_bs_corr_crop.isig[1].data
)

# Rotate (helix correction)

In [ ]:
deg = 30 # Kanskje 90 grader feil
rot = np.exp(1j * np.deg2rad(deg))

In [ ]:
s_bs_corr_crop_rot = s_bs_corr_crop * rot

# Renaming

In [ ]:
data = s_bs_corr_crop_rot

## Lage disk maske for testing

In [ ]:
center = (21, 18)
radius = 8
x = np.array([[i for i in range(1, data.shape[0]+1)] for _ in range(data.shape[1])])
y = np.array([[i for _ in range(data.shape[0])] for i in range(1, data.shape[1]+1)])
mask = (x-center[0]-1)**2+(y-center[1]-1)**2<=radius**2

In [ ]:
ax.imshow(np.abs(data*mask), cmap="grey")

Now we have `data` and `mask`

# Normalization

In [ ]:
data_masked = data*mask

In [ ]:
def normalize(img):
    axis = (-2, -1)
    img = img-img.min()
    img = img/img.max()
    return img*2-1

In [ ]:
real_norm = normalize(data_masked.real)
imag_norm = normalize(data_masked.imag)*1j
data_masked_norm = real_norm+imag_norm

In [ ]:
ax.imshow(data_masked_norm.real, cmap="grey")

# Renaming

In [ ]:
data = data_masked_norm

# Root finding

In [ ]:
x = np.arange(data.shape[0])
y = np.arange(data.shape[1])

real_interp = RegularGridInterpolator((x, y), data.real, method="linear")
imag_interp = RegularGridInterpolator((x, y), data.imag, method="linear")

# Need to use item because interpolator returns 0 dim numpy array
# The use of int here will probably cause problems later, should
# round toward the center of the circle instead.
def objective(x):
    if not mask[int(x[0]), int(x[1])]:
        return 2
    a = real_interp(x).item()
    b = imag_interp(x).item()
    return (a, b)

print(objective((center[1], center[0])))

In [ ]:
# May need to change root finding algorithm, but default worked this time
result = root(objective, (center[1], center[0]))
result

In [ ]:
objective(result.x)